/Volumes/workspace/default/my_volume/netflix_titles.csv

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Netflix") \
    .getOrCreate()

df_practice = spark.read \
    .format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load("/Volumes/workspace/default/my_volume/netflix_titles.csv")

In [0]:
df_practice.groupBy("type").count().show()

+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2676|
|        Movie| 6131|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



In [0]:
df_practice.groupBy("country") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------------------+-----+
|             country|count|
+--------------------+-----+
|       United States| 2805|
|               India|  972|
|                NULL|  832|
|      United Kingdom|  419|
|               Japan|  245|
|         South Korea|  199|
|              Canada|  181|
|               Spain|  145|
|              France|  123|
|              Mexico|  110|
|               Egypt|  106|
|              Turkey|  105|
|             Nigeria|   93|
|           Australia|   87|
|              Taiwan|   81|
|           Indonesia|   79|
|              Brazil|   77|
|United Kingdom, U...|   75|
|         Philippines|   75|
|United States, Ca...|   73|
+--------------------+-----+
only showing top 20 rows


In [0]:
df_practice.groupBy("director") \
    .count() \
    .orderBy(col("count").desc()) \
    .limit(5) \
    .show()
 

+--------------------+-----+
|            director|count|
+--------------------+-----+
|                NULL| 2636|
|       Rajiv Chilaka|   19|
|Raúl Campos, Jan ...|   18|
|        Marcus Raboy|   16|
|         Suhas Kadav|   16|
+--------------------+-----+



In [0]:
df_practice.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
#convert date_added to proper date type
 #I want to  extract numeric value from duration 90-min-90(numeric)
from pyspark.sql.functions import col, trim, to_date

df_clean = df_practise.withColumn(
    "date_added",
    to_date(trim(col("date_added")), "MMMM d, yyyy")
)
        #for eg if your date is september 10,2021
        #\\d+ for eg : 90 min,


In [0]:
df_clean.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: date (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
df_clean = df_practice.withColumn(
    "date_added",
    expr("try_to_date(trim(date_added), 'MMMM d, yyyy')")
)

df_clean.select(
    "title",
    "date_added",
    "duration"
).show(5)

+--------------------+----------+---------+
|               title|date_added| duration|
+--------------------+----------+---------+
|Dick Johnson Is Dead|2021-09-25|   90 min|
|       Blood & Water|2021-09-24|2 Seasons|
|           Ganglands|2021-09-24| 1 Season|
|Jailbirds New Orl...|2021-09-24| 1 Season|
|        Kota Factory|2021-09-24|2 Seasons|
+--------------------+----------+---------+
only showing top 5 rows


In [0]:
#col(c).isNUll will return True/False for each row
# cast("int") will convert into True- make it into 1 and False- make it to 0.
#sum will give all the result

df_practice.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_practice.columns
]).show()

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   1|    2|    2636| 826|    832|        13|           2|     6|       5|        3|          3|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [0]:
df_practice.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [0]:
# remove rows with any null values
df_no_nulls = df_practice.dropna()

df_no_nulls.show()

+-------+-------+--------------------+-------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|           director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+-------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s8|  Movie|             Sankofa|       Haile Gerima|Kofi Ghanaba, Oya...|United States, Gh...|September 24, 2021|        1993| TV-MA|  125 min|Dramas, Independe...|On a photo shoot ...|
|     s9|TV Show|The Great British...|    Andy Devonshire|Mel Giedroyc, Sue...|      United Kingdom|September 24, 2021|        2021| TV-14|9 Seasons|British TV Shows,...|A talented batch ...|
|    s10|  Movie|        The Starling|  

In [0]:
df_filtered= df_practice.dropna(
    subset = ["director", "country"]
)
df_filtered.show()

+-------+-------+--------------------+-------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|           director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+-------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|    Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s8|  Movie|             Sankofa|       Haile Gerima|Kofi Ghanaba, Oya...|United States, Gh...|September 24, 2021|        1993| TV-MA|  125 min|Dramas, Independe...|On a photo shoot ...|
|     s9|TV Show|The Great British...|  

In [0]:
#remove row where all values are null

df_clean1 = df_practice.dropna(how="all")
#remove rows where all values are null
df_filled = df_practice.fillna({
    "director" : "Unknown",
    "country" : "Not available",
    "Release_year" : 0 
})
df_filled.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|             Unknown|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan